# DSFB-DSCD threshold sweep replay

Generate a run directory from the workspace root with:

```bash
cargo run --release -p dsfb-dscd -- --num-events 8192 --tau-steps 201
```

By default, the next cell tries several ways to locate DSCD outputs and auto-selects the newest run.
You can force paths with environment variables:
- `DSFB_DSCD_RUN_DIR`: absolute path to one specific run folder.
- `DSFB_DSCD_OUTPUT_ROOT`: absolute path to the `output-dsfb-dscd` folder.
- `DSFB_WORKSPACE_ROOT`: absolute path to the DSFB workspace root.



In [ ]:
from pathlib import Path
from typing import List, Optional
import os


def _is_workspace_root(path: Path) -> bool:
    return (path / "Cargo.toml").exists() and (path / "crates" / "dsfb-dscd").exists()


def _discover_workspace_root() -> Optional[Path]:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if _is_workspace_root(candidate):
            return candidate

    search_roots = [
        Path.home(),
        Path("/home"),
        Path("/content"),
        Path("/workspace"),
        Path("/workspaces"),
        Path("/kaggle/working"),
    ]
    for root in search_roots:
        if not root.exists():
            continue

        if _is_workspace_root(root):
            return root

        for pattern in ("*/Cargo.toml", "*/*/Cargo.toml", "*/*/*/Cargo.toml", "*/*/*/*/Cargo.toml"):
            for cargo_toml in root.glob(pattern):
                candidate = cargo_toml.parent
                if _is_workspace_root(candidate):
                    return candidate

    return None


def _dedupe_paths(paths: List[Path]) -> List[Path]:
    out: List[Path] = []
    seen = set()
    for path in paths:
        key = str(path)
        if key in seen:
            continue
        seen.add(key)
        out.append(path)
    return out


def _is_run_dir(path: Path) -> bool:
    return path.is_dir() and (path / "threshold_sweep.csv").exists()


def _candidate_output_roots(workspace_root: Optional[Path]) -> List[Path]:
    env_workspace = os.environ.get("DSFB_WORKSPACE_ROOT")
    env_output_root = os.environ.get("DSFB_DSCD_OUTPUT_ROOT")

    roots: List[Path] = []

    if env_output_root:
        roots.append(Path(env_output_root).expanduser())

    if env_workspace:
        roots.append(Path(env_workspace).expanduser() / "output-dsfb-dscd")

    if workspace_root is not None:
        roots.append(workspace_root / "output-dsfb-dscd")

    cwd = Path.cwd().resolve()
    roots.append(cwd / "output-dsfb-dscd")
    roots.extend(parent / "output-dsfb-dscd" for parent in cwd.parents)

    roots.extend([
        Path.home() / "output-dsfb-dscd",
        Path("/content/output-dsfb-dscd"),
        Path("/content/dsfb/output-dsfb-dscd"),
        Path("/workspace/output-dsfb-dscd"),
        Path("/workspaces/output-dsfb-dscd"),
    ])

    return _dedupe_paths([path.resolve() for path in roots])


WORKSPACE_ROOT = _discover_workspace_root()
OUTPUT_ROOTS = _candidate_output_roots(WORKSPACE_ROOT)

RUN_DIR = os.environ.get("DSFB_DSCD_RUN_DIR")

if RUN_DIR is not None:
    RUN_DIR = Path(RUN_DIR).expanduser().resolve()
    if not _is_run_dir(RUN_DIR):
        raise FileNotFoundError(
            f"Configured DSFB_DSCD_RUN_DIR is not a valid DSCD run folder: {RUN_DIR}. "
            "Expected threshold_sweep.csv inside that directory."
        )
else:
    selected = None
    selected_output_root = None

    for root in OUTPUT_ROOTS:
        if _is_run_dir(root):
            selected = root
            selected_output_root = root.parent
            break

        if not root.exists() or not root.is_dir():
            continue

        run_candidates = sorted(path for path in root.iterdir() if path.is_dir())
        if run_candidates:
            selected = run_candidates[-1]
            selected_output_root = root
            break

    if selected is None:
        searched = "\n".join(f" - {path}" for path in OUTPUT_ROOTS)
        raise FileNotFoundError(
            "No DSCD output folders found. Set DSFB_DSCD_RUN_DIR to a specific run folder, "
            "or DSFB_DSCD_OUTPUT_ROOT to an output root. Searched:\n" + searched
        )

    RUN_DIR = selected
    OUTPUT_ROOT = selected_output_root

print(f"Workspace root: {WORKSPACE_ROOT}")
print(f"Output roots searched: {len(OUTPUT_ROOTS)}")
print(f"Using RUN_DIR: {RUN_DIR}")
RUN_DIR


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

threshold_df = pd.read_csv(RUN_DIR / 'threshold_sweep.csv')
threshold_df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(threshold_df['tau'], threshold_df['expansion_ratio'], linewidth=2)
ax.set_title('DSCD expansion ratio vs trust threshold')
ax.set_xlabel('tau')
ax.set_ylabel('expansion_ratio')
ax.grid(alpha=0.25)
plt.show()

In [ ]:
finite_size_path = RUN_DIR / 'finite_size_scaling.csv'
if finite_size_path.exists():
    finite_df = pd.read_csv(finite_size_path)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(finite_df['num_events'], finite_df['transition_width'], marker='o')
    axes[0].set_title('Transition width vs N')
    axes[0].set_xlabel('num_events')
    axes[0].set_ylabel('transition_width')
    axes[0].grid(alpha=0.25)

    axes[1].plot(finite_df['num_events'], finite_df['max_derivative'], marker='o')
    axes[1].set_title('Max derivative vs N')
    axes[1].set_xlabel('num_events')
    axes[1].set_ylabel('max_derivative')
    axes[1].grid(alpha=0.25)
    plt.show()
else:
    print('finite_size_scaling.csv not present for this run')

In [ ]:
events_path = RUN_DIR / 'graph_events.csv'
edges_path = RUN_DIR / 'graph_edges.csv'

events_df = pd.read_csv(events_path)
edges_df = pd.read_csv(edges_path)

subset_edges = edges_df.head(128)
subset_nodes = sorted(set(subset_edges['from_event_id']).union(subset_edges['to_event_id']))

graph = nx.DiGraph()
graph.add_nodes_from(subset_nodes)
graph.add_edges_from(subset_edges[['from_event_id', 'to_event_id']].itertuples(index=False, name=None))

plt.figure(figsize=(10, 6))
pos = nx.spring_layout(graph, seed=7)
nx.draw_networkx(graph, pos=pos, node_size=120, with_labels=False, arrows=True)
plt.title('DSCD graph preview')
plt.axis('off')
plt.show()